In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Portes fractionnaires

<details>
<summary><b>Versions des paquets</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Cette page présente deux nouveaux types de portes désormais pris en charge sur la flotte de QPUs IBM Quantum&reg;. Ces portes *fractionnaires* sont disponibles sur les [QPUs Heron](/guides/processor-types#heron) sous la forme suivante :
- $R_{ZZ}(\theta)$ pour $0 \lt \theta \leq \pi/2$
- $R_X(\theta)$ pour tout $\theta$

Cette page aborde les cas d'usage dans lesquels l'implémentation de portes fractionnaires peut améliorer l'efficacité de tes workflows, ainsi que la manière d'utiliser ces portes sur les QPUs IBM Quantum.

## Comment utiliser les portes fractionnaires
En interne, ces portes fractionnaires fonctionnent en exécutant directement une rotation $R_{ZZ}(\theta)$ et $R_X(\theta)$ pour un angle arbitraire. L'utilisation de la porte $R_X(\theta)$ peut réduire la durée et l'erreur des rotations à un seul qubit d'angle arbitraire par un facteur pouvant aller jusqu'à deux. L'exécution directe de la rotation de la porte $R_{ZZ}(\theta)$ évite la décomposition en plusieurs objets `CZGate`, réduisant ainsi la durée et l'erreur d'un circuit. C'est particulièrement utile pour les circuits qui contiennent de nombreuses rotations à un et deux qubits, comme lors de la simulation de la dynamique d'un système quantique ou lors de l'utilisation d'un ansatz variationnel avec de nombreux paramètres.

Bien que ces types de portes fassent partie de la [bibliothèque de portes standard](./circuit-library) qu'un `QuantumCircuit` peut posséder, elles ne peuvent être utilisées que sur des QPUs IBM Quantum spécifiques, et doivent être chargées avec l'option `use_fractional_gates` définie à `True` (comme indiqué ci-dessous). Cette option garantit que les portes fractionnaires sont incluses dans le `Target` du backend pour le Transpiler.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization.timeline import draw as draw_timeline, IQXSimple

from qiskit_ibm_runtime import QiskitRuntimeService


num_qubits = 5
num_time_steps = 3
rx_angle = 0.1
rzz_angle = 0.1

ising_circuit = QuantumCircuit(num_qubits)
for i in range(num_time_steps):
    # rx layer
    for q in range(num_qubits):
        ising_circuit.rx(rx_angle, q)
    for q in range(1, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    # 2nd rzz layer
    for q in range(0, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    ising_circuit.barrier()
ising_circuit.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg" alt="Output of the previous code cell" />

Cet exemple de code montre comment utiliser les portes fractionnaires dans le cadre d'un workflow qui simule la dynamique d'une chaîne d'Ising avec des portes fractionnaires. La durée du circuit est ensuite comparée à celle d'un backend qui n'utilise pas de portes fractionnaires.

> **Note:** La valeur d'erreur rapportée dans le `Target` d'un backend avec les portes fractionnaires activées n'est qu'une copie de l'équivalent de la porte non fractionnaire (qui peut ne pas être identique). En effet, la communication des taux d'erreur pour les portes fractionnaires n'est pas encore prise en charge.

  Cependant, étant donné que le temps de porte des portes fractionnaires et non fractionnaires est identique, il est raisonnable de supposer que leurs taux d'erreur sont comparables — en particulier lorsque la principale source d'erreur dans un circuit est due à la relaxation.

In [2]:
service = QiskitRuntimeService()
backend_fractional = service.backend("ibm_torino", use_fractional_gates=True)
backend_conventional = service.backend(
    "ibm_torino", use_fractional_gates=False
)

pm_fractional = generate_preset_pass_manager(
    optimization_level=3, backend=backend_fractional, scheduling_method="alap"
)
pm_conventional = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend_conventional,
    scheduling_method="alap",
)

ising_circuit_fractional = pm_fractional.run(ising_circuit)
ising_circuit_conventional = pm_conventional.run(ising_circuit)

![Output of the previous code cell](../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg)

Spécifie deux objets backend : l'un avec les portes fractionnaires activées, et l'autre avec elles désactivées, puis transpile-les tous les deux.

In [3]:
# Draw timeline of circuit with conventional gates
draw_timeline(
    ising_circuit_conventional,
    idle_wires=False,
    target=backend_conventional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/0013f5fa-4072-4aa4-94fe-7e195435f828-0.svg" alt="Output of the previous code cell" />

In [4]:
# Draw timeline of circuit with fractional gates
draw_timeline(
    ising_circuit_fractional,
    idle_wires=False,
    target=backend_fractional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/08dd1cdf-8b34-47c2-8324-f3538c9d1ab6-0.svg" alt="Output of the previous code cell" />

### Angle constraints

For the two-qubit $R_{ZZ}(\theta)$ gate, only angles between $0$ and $\pi/2$ can be executed on IBM Quantum hardware. If a circuit contains any $R_{ZZ}(\theta)$ gates with an angle outside of this range, then the standard transpilation pipeline will generally correct this with an appropriate circuit transformation (through the [`FoldRzzAngle`](../api/qiskit-ibm-runtime/transpiler-passes-fold-rzz-angle) pass).  However, for any $R_{ZZ}(\theta)$ gate containing one or more [`Parameter`](../api/qiskit/qiskit.circuit.Parameter)s, the transpiler will assume these parameters will be assigned angles within this range at runtime. The job will fail if any of the parameter values specified in the PUB submitted to Qiskit Runtime are outside of this range.

## Where to use fractional gates

Historically, the basis gates available on IBM Quantum QPUs have been **`CZ`**, **`X`**, **`RZ`**, **`SX`**, and **`ID`**, which can not efficiently represent circuits with single- and two-qubit rotations that are not multiples of $\pi / 2$. For example, an $R_X(\theta)$ gate, when transpiled, must decompose into a series of $RZ$ and $\sqrt{X}$ gates, which creates a circuit with two gates of finite duration instead of one.

Similarly, when two-qubit rotations such as an $R_{ZZ}(\theta)$ gate are transpiled, the decomposition requires two **`CZ`** gates and several single-qubit gates, which increases the circuit depth. These decompositions are shown in the following code.

In [5]:
qc = QuantumCircuit(1)
param = Parameter("θ")
qc.rx(param, 0)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/11b003ee-aa8e-4794-a84a-b6870d15fa11-0.svg" alt="Output of the previous code cell" />

In [6]:
# Decomposition of an RX(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/5da9bba4-9a3b-4569-9997-c5b9ccf87b6a-0.svg" alt="Output of the previous code cell" />

In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService

qc = QuantumCircuit(2)
param = Parameter("θ")
qc.rzz(param, 0, 1)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/6b256201-0237-4f63-91ff-fde1bb884804-0.svg" alt="Output of the previous code cell" />

In [8]:
# Decomposition of an RZZ(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/f07217b9-a6f0-4adf-b341-6da447535c33-0.svg" alt="Output of the previous code cell" />

For workflows that require many single-qubit $R_X(\theta)$ or two-qubit rotations (such as in a variational ansatz or when simulating the time evolution of quantum systems), this constraint causes the circuit depth to grow rapidly. However, fractional gates remove this requirement, because the single- and two-qubit rotations are executed directly, and create a more efficient (and thus error-suppressed) quantum circuit.

<span id="when-not-to-use"></span>
## When *not* to use fractional gates

It is important to note that fractional gates are an *experimental* feature and the behavior of the `use_fractional_gates` flag may change in the future. Look to the [release notes](/docs/api/qiskit-ibm-runtime/release-notes) for new versions of Qiskit Runtime for more information. See also the API reference documentation for [QiskitRuntimeService.backend](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#backend), which describes `use_fractional_gates`.

Additionally, the Qiskit transpiler has limited capability to use $R_{ZZ}(\theta)$ in its optimization passes. This requires you to take more care in crafting and optimizing circuits that contain these instructions.

Lastly, using fractional gates is not supported for:
- [Dynamic circuits](/docs/guides/classical-feedforward-and-control-flow)
- [Pauli twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) - however, [measurement twirling with TREX](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) *is* supported.
- [Probabilistic error cancellation](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)
- [Zero-noise extrapolation (using probabilistic error amplification)](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea)

Read the guide on [primitive options](/docs/guides/runtime-options-overview) to learn more about customizing the error mitigation and suppression techniques for a given quantum workload.

## Next steps

<Admonition type="tip" title="Recommendations">
  -  To learn more about transpilation, see the [introduction to transpilation](/docs/guides/transpile) page.
  -  Read about [writing a custom transpiler pass](/docs/guides/custom-transpiler-pass).
  -  Understand how to [configure error mitigation](/docs/guides/configure-error-mitigation) for Qiskit Runtime.
</Admonition>